In [ ]:
import os, warnings, random
warnings.filterwarnings("ignore")

# Paths
CLOSE_DIR = ""   # monitor traffic
OPEN_DIR  = ""   # unmonitor traffic

# Sequence
MAXLEN      = 5000
NB_FEATURES = 1

# Unknown (open-world) sample counts
N_UNKNOWN_TRAIN = 400
N_UNKNOWN_TEST  = 10000

# Training
BATCH_SIZE = 128
EPOCHS     = 100
PATIENCE   = 10

# Reproducibility
SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)

import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_curve, auc,
                             precision_recall_curve, average_precision_score)
import pandas as pd
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, Conv1D, MaxPooling1D,
                                     Flatten, LSTM, Input)
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


def extract_direction(filepath):
    """
    Read a trace file and return a direction sequence.
    Each line: <timestamp> <signed_packet_size>
    direction = +1 if size > 0 else -1. Pads/truncates to MAXLEN.
    Returns numpy array of shape (MAXLEN,).
    """
    directions = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            try:
                size = float(parts[1])
                directions.append(1.0 if size > 0 else -1.0)
            except ValueError:
                continue

    directions = directions[:MAXLEN]
    if len(directions) < MAXLEN:
        directions += [0.0] * (MAXLEN - len(directions))
    return np.array(directions, dtype=np.float32)


def load_dataset(root_dir, label_override=None):
    """
    Walk root_dir/ where subfolders are class labels.
    label_override: if set, every sample gets this label (for open/unknown).
    Returns X (N, MAXLEN), y (N,).
    """
    X, y = [], []
    for label_dir in sorted(os.listdir(root_dir)):
        full_label_path = os.path.join(root_dir, label_dir)
        if not os.path.isdir(full_label_path):
            continue
        label = label_override if label_override is not None else label_dir
        for fp in glob.glob(os.path.join(full_label_path, "*.txt")):
            X.append(extract_direction(fp))
            y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)


# Load datasets
X_close, y_close = load_dataset(CLOSE_DIR, label_override=None)
X_open,  y_open  = load_dataset(OPEN_DIR,  label_override="unknown")

# Closed-world split
le_closed = LabelEncoder()
y_close_enc = le_closed.fit_transform(y_close)
N_CLASSES_CLOSED = len(le_closed.classes_)

X_train_cw, X_test_cw, y_train_cw, y_test_cw = train_test_split(
    X_close, y_close_enc,
    test_size=0.2, random_state=SEED, stratify=y_close_enc
)

y_train_cw_cat = to_categorical(y_train_cw, N_CLASSES_CLOSED)
y_test_cw_cat  = to_categorical(y_test_cw,  N_CLASSES_CLOSED)

X_train_cw_3d = X_train_cw.reshape(-1, MAXLEN, NB_FEATURES)
X_test_cw_3d  = X_test_cw.reshape( -1, MAXLEN, NB_FEATURES)

# Open-world split
idx_all_open = np.random.permutation(len(X_open))

need = N_UNKNOWN_TRAIN + N_UNKNOWN_TEST
assert len(idx_all_open) >= need, (
    f"Not enough open samples! Have {len(idx_all_open)}, need {need}."
)

X_unk_train = X_open[idx_all_open[:N_UNKNOWN_TRAIN]]
X_unk_test  = X_open[idx_all_open[N_UNKNOWN_TRAIN:need]]

UNKNOWN_LABEL  = N_CLASSES_CLOSED
N_CLASSES_OPEN = N_CLASSES_CLOSED + 1

X_train_ow = np.concatenate([X_train_cw, X_unk_train], axis=0)
y_train_ow = np.concatenate([y_train_cw, np.full(N_UNKNOWN_TRAIN, UNKNOWN_LABEL)], axis=0)

X_test_ow  = np.concatenate([X_test_cw, X_unk_test],  axis=0)
y_test_ow  = np.concatenate([y_test_cw, np.full(N_UNKNOWN_TEST, UNKNOWN_LABEL)],  axis=0)

perm_tr = np.random.permutation(len(X_train_ow))
X_train_ow, y_train_ow = X_train_ow[perm_tr], y_train_ow[perm_tr]

perm_te = np.random.permutation(len(X_test_ow))
X_test_ow, y_test_ow   = X_test_ow[perm_te],  y_test_ow[perm_te]

y_train_ow_cat = to_categorical(y_train_ow, N_CLASSES_OPEN)
y_test_ow_cat  = to_categorical(y_test_ow,  N_CLASSES_OPEN)

X_train_ow_3d = X_train_ow.reshape(-1, MAXLEN, NB_FEATURES)
X_test_ow_3d  = X_test_ow.reshape( -1, MAXLEN, NB_FEATURES)


def get_callbacks(filepath):
    return [
        EarlyStopping(monitor="val_loss", patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(filepath, monitor="val_loss",
                        save_best_only=True, verbose=0),
    ]


def evaluate_closed(model, X_test, y_test, y_test_cat, nb_classes, tag):
    """Evaluate model in closed-world setting and print classification report."""
    y_pred = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE), axis=1)
    acc = accuracy_score(y_test, y_pred)
    print(f"{tag} | Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, digits=4))
    return acc


def evaluate_open(model, X_test, y_test, y_test_cat,
                  nb_classes, unknown_label, tag, prefix):
    """
    Evaluate model in open-world setting.
    Saves {prefix}_precision_recall_curve.csv and {prefix}_roc_curve.csv.
    """
    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE)
    y_pred      = np.argmax(y_pred_prob, axis=1)
    acc = accuracy_score(y_test, y_pred)
    print(f"{tag} | Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, digits=4))

    # Binary: is sample "unknown"?
    y_bin_true  = (y_test == unknown_label).astype(int)
    y_bin_score = y_pred_prob[:, unknown_label]

    N_POINTS = 1000

    # Precision-recall curve
    prec, rec, pr_thresh = precision_recall_curve(y_bin_true, y_bin_score)
    ap = average_precision_score(y_bin_true, y_bin_score)
    pr_idx = np.linspace(0, len(pr_thresh) - 1, min(N_POINTS, len(pr_thresh)), dtype=int)
    pd.DataFrame({
        "precision": prec[pr_idx],
        "recall":    rec[pr_idx],
        "threshold": pr_thresh[pr_idx],
    }).to_csv(f"{prefix}_precision_recall_curve.csv", index=False)

    # ROC curve
    fpr, tpr, roc_thresh = roc_curve(y_bin_true, y_bin_score)
    roc_auc = auc(fpr, tpr)
    roc_idx = np.linspace(0, len(roc_thresh) - 1, min(N_POINTS, len(roc_thresh)), dtype=int)
    pd.DataFrame({
        "fpr":       fpr[roc_idx],
        "tpr":       tpr[roc_idx],
        "threshold": roc_thresh[roc_idx],
    }).to_csv(f"{prefix}_roc_curve.csv", index=False)

    print(f"Average Precision: {ap:.4f} | ROC-AUC: {roc_auc:.4f}")
    return acc

In [ ]:

# ── CNN layer config (mirrors code-original dict structure) ──
CNN_LAYERS = [
    {"name": "conv",       "filters": 32,  "kernel_size": 8, "activation": "relu",    "stride": 1},
    {"name": "conv",       "filters": 64,  "kernel_size": 8, "activation": "relu",    "stride": 1},
    {"name": "maxpooling", "pool_size": 8},
    {"name": "dropout",    "rate": 0.25},
    {"name": "flatten"},
    {"name": "dense",      "units": 512,   "activation": "relu",  "regularization": 0.0},
    {"name": "dropout",    "rate": 0.5},
]

def build_cnn(learn_params, nb_classes):
    """
    Rebuild CNN from layer-config dict list.
    Equivalent to the original build_model() for CNN.
    """
    maxlen   = learn_params["maxlen"]
    nb_feat  = learn_params["nb_features"]
    layers   = learn_params["layers"]

    model = Sequential()
    first = True

    for l in layers:
        name = l["name"]
        if name == "conv":
            if first:
                model.add(Conv1D(
                    filters=l["filters"],
                    kernel_size=l["kernel_size"],
                    padding="valid",
                    activation=l["activation"],
                    strides=l["stride"],
                    input_shape=(maxlen, nb_feat)
                ))
                first = False
            else:
                model.add(Conv1D(
                    filters=l["filters"],
                    kernel_size=l["kernel_size"],
                    padding="valid",
                    activation=l["activation"],
                    strides=l["stride"],
                ))
        elif name == "maxpooling":
            model.add(MaxPooling1D(pool_size=l["pool_size"], padding="valid"))
        elif name == "dropout":
            if first:
                model.add(Dropout(rate=l["rate"],
                                  input_shape=(maxlen, nb_feat)))
                first = False
            else:
                model.add(Dropout(rate=l["rate"]))
        elif name == "flatten":
            model.add(Flatten())
        elif name == "dense":
            reg = l.get("regularization", 0.0)
            if reg > 0.0:
                model.add(Dense(units=l["units"], activation=l["activation"],
                                kernel_regularizer=regularizers.l2(reg),
                                activity_regularizer=regularizers.l1(reg)))
            else:
                model.add(Dense(units=l["units"], activation=l["activation"]))

    # Final softmax
    model.add(Dense(units=nb_classes, activation="softmax"))
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model


CNN_PARAMS = {"maxlen": MAXLEN, "nb_features": NB_FEATURES, "layers": CNN_LAYERS}

# ── Phase 1: CNN Closed-World ──────────────────────────────────────────────────
print("\n" + "█"*60)
print("  CNN  —  CLOSED WORLD")
print("█"*60)

cnn_cw = build_cnn(CNN_PARAMS, N_CLASSES_CLOSED)
cnn_cw.summary()

cnn_cw.fit(
    X_train_cw_3d, y_train_cw_cat,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_split=0.1,
    callbacks=get_callbacks("cnn_closed_best.h5"),
    verbose=1,
)
evaluate_closed(cnn_cw, X_test_cw_3d, y_test_cw, y_test_cw_cat,
                N_CLASSES_CLOSED, "CNN")

# ── Phase 2: CNN Open-World ────────────────────────────────────────────────────
print("\n" + "█"*60)
print("  CNN  —  OPEN WORLD  (N+1 classes)")
print("█"*60)

cnn_ow = build_cnn(CNN_PARAMS, N_CLASSES_OPEN)

cnn_ow.fit(
    X_train_ow_3d, y_train_ow_cat,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_split=0.1,
    callbacks=get_callbacks("cnn_open_best.h5"),
    verbose=1,
)
evaluate_open(cnn_ow, X_test_ow_3d, y_test_ow, y_test_ow_cat,
              N_CLASSES_OPEN, UNKNOWN_LABEL, "CNN",
              prefix="tiktok")

In [ ]:

# ── LSTM layer config ──
LSTM_LAYERS = [
    {"units": 128, "activation": "tanh", "rec_activation": "sigmoid", "dropout": 0.1},
    {"units": 64,  "activation": "tanh", "rec_activation": "sigmoid", "dropout": 0.1},
]

def build_lstm(learn_params, nb_classes):
    """
    Equivalent to original LSTM build_model().
    Supports single-layer and multi-layer configs.
    """
    maxlen  = learn_params["maxlen"]
    nb_feat = learn_params["nb_features"]
    layers  = learn_params["layers"]

    model = Sequential()

    if len(layers) == 1:
        l = layers[0]
        model.add(LSTM(
            units=l["units"],
            activation=l["activation"],
            recurrent_activation=l["rec_activation"],
            dropout=l["dropout"],
            return_sequences=False,
            input_shape=(maxlen, nb_feat),
        ))
        model.add(Dense(units=nb_classes, activation="softmax"))
        model.compile(optimizer=Adam(learning_rate=1e-3),
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])
        return model

    # Multi-layer
    first_l  = layers[0]
    last_l   = layers[-1]
    middle_ls = layers[1:-1]

    model.add(LSTM(
        units=first_l["units"],
        activation=first_l["activation"],
        recurrent_activation=first_l["rec_activation"],
        dropout=first_l["dropout"],
        return_sequences=True,
        input_shape=(maxlen, nb_feat),
    ))
    for l in middle_ls:
        model.add(LSTM(
            units=l["units"],
            activation=l["activation"],
            recurrent_activation=l["rec_activation"],
            dropout=l["dropout"],
            return_sequences=True,
        ))
    model.add(LSTM(
        units=last_l["units"],
        activation=last_l["activation"],
        recurrent_activation=last_l["rec_activation"],
        dropout=last_l["dropout"],
        return_sequences=False,
    ))
    model.add(Dense(units=nb_classes, activation="softmax"))
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model


LSTM_PARAMS = {"maxlen": MAXLEN, "nb_features": NB_FEATURES, "layers": LSTM_LAYERS}

# ── Phase 1: LSTM Closed-World ─────────────────────────────────────────────────
print("\n" + "█"*60)
print("  LSTM  —  CLOSED WORLD")
print("█"*60)

lstm_cw = build_lstm(LSTM_PARAMS, N_CLASSES_CLOSED)
lstm_cw.summary()

lstm_cw.fit(
    X_train_cw_3d, y_train_cw_cat,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_split=0.1,
    callbacks=get_callbacks("lstm_closed_best.h5"),
    verbose=1,
)
evaluate_closed(lstm_cw, X_test_cw_3d, y_test_cw, y_test_cw_cat,
                N_CLASSES_CLOSED, "LSTM")

# ── Phase 2: LSTM Open-World ───────────────────────────────────────────────────
print("\n" + "█"*60)
print("  LSTM  —  OPEN WORLD  (N+1 classes)")
print("█"*60)

lstm_ow = build_lstm(LSTM_PARAMS, N_CLASSES_OPEN)

lstm_ow.fit(
    X_train_ow_3d, y_train_ow_cat,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_split=0.1,
    callbacks=get_callbacks("lstm_open_best.h5"),
    verbose=1,
)
evaluate_open(lstm_ow, X_test_ow_3d, y_test_ow, y_test_ow_cat,
              N_CLASSES_OPEN, UNKNOWN_LABEL, "LSTM",
              prefix="tiktok_lstm")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════

# ── SDAE layer config ──
# in_dim must match MAXLEN (flat input)
SDAE_LAYERS = [
    {"in_dim": MAXLEN, "out_dim": 1000, "enc_activation": "tanh", "dec_activation": "tanh",
     "dropout": 0.2, "epochs": 20, "batch_size": 128,
     "optimizer": "adam", "adam": {"lr": 1e-3, "decay": 0.0}},
    {"in_dim": 1000,   "out_dim": 500,  "enc_activation": "tanh", "dec_activation": "tanh",
     "dropout": 0.3, "epochs": 20, "batch_size": 128,
     "optimizer": "adam", "adam": {"lr": 1e-3, "decay": 0.0}},
]


def make_ae_layer(layer_cfg, x_train_np, x_test_np):
    """
    Pre-train a single AE layer.
    Returns:
        new_x_train, new_x_test  (encoded representations)
        weights                  (encoder layer weights to transfer)
    """
    in_dim   = layer_cfg["in_dim"]
    out_dim  = layer_cfg["out_dim"]
    enc_act  = layer_cfg["enc_activation"]
    dec_act  = layer_cfg["dec_activation"]
    epochs   = layer_cfg["epochs"]
    bs       = layer_cfg["batch_size"]
    lr       = layer_cfg["adam"]["lr"]
    decay    = layer_cfg["adam"]["decay"]

    inp     = Input(shape=(in_dim,))
    encoded = Dense(out_dim, activation=enc_act)(inp)
    decoded = Dense(in_dim,  activation=dec_act)(encoded)

    autoencoder = Model(inp, decoded)
    encoder     = Model(inp, encoded)
    autoencoder.compile(optimizer=Adam(learning_rate=lr, decay=decay),
                        loss="mean_squared_error")
    autoencoder.fit(x_train_np, x_train_np,
                    epochs=epochs, batch_size=bs, verbose=0)

    new_x_train = encoder.predict(x_train_np, batch_size=bs, verbose=0)
    new_x_test  = encoder.predict(x_test_np,  batch_size=bs, verbose=0)
    weights     = encoder.layers[1].get_weights()   # Dense weights
    return new_x_train, new_x_test, weights


def build_sdae(layer_cfgs, nb_classes,
               x_train_flat, x_test_flat,
               pre_train=True):
    """
    Build SAE with optional greedy layer-wise pre-training.
    Input  : flat vectors (N, MAXLEN)
    Output : Keras Model with softmax head
    """
    # ── Build full model and keep direct references to encoder Dense layers ──
    inp      = Input(shape=(layer_cfgs[0]["in_dim"],))
    prev     = inp
    enc_dense_layers = []   # direct Keras layer objects

    for cfg in layer_cfgs:
        enc_layer = Dense(cfg["out_dim"], activation=cfg["enc_activation"])
        enc       = enc_layer(prev)
        enc_dense_layers.append(enc_layer)   # store layer object, not index

        drop = cfg.get("dropout", 0.0)
        if drop > 0.0:
            enc = Dropout(drop)(enc)
        prev = enc

    out   = Dense(nb_classes, activation="softmax")(prev)
    model = Model(inp, out)

    if pre_train:
        print("  SDAE: greedy layer-wise pre-training …")
        cur_x_train = x_train_flat.copy()
        cur_x_test  = x_test_flat.copy()

        for i, cfg in enumerate(layer_cfgs):
            print(f"    Pre-training AE layer {i+1}: "
                  f"{cfg['in_dim']} → {cfg['out_dim']}")
            cur_x_train, cur_x_test, weights = make_ae_layer(
                cfg, cur_x_train, cur_x_test
            )
            # Transfer weights via direct layer reference (safe, no index math)
            enc_dense_layers[i].set_weights(weights)

    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model


# Flat (2D) inputs for SDAE
X_train_cw_flat = X_train_cw          # (N, MAXLEN)
X_test_cw_flat  = X_test_cw

X_train_ow_flat = X_train_ow
X_test_ow_flat  = X_test_ow

# ── Phase 1: SDAE Closed-World ─────────────────────────────────────────────────
print("\n" + "█"*60)
print("  SDAE  —  CLOSED WORLD")
print("█"*60)

sdae_cw = build_sdae(SDAE_LAYERS, N_CLASSES_CLOSED,
                     X_train_cw_flat, X_test_cw_flat,
                     pre_train=True)
sdae_cw.summary()

sdae_cw.fit(
    X_train_cw_flat, y_train_cw_cat,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_split=0.1,
    callbacks=get_callbacks("sdae_closed_best.h5"),
    verbose=1,
)
evaluate_closed(sdae_cw, X_test_cw_flat, y_test_cw, y_test_cw_cat,
                N_CLASSES_CLOSED, "SDAE")

# ── Phase 2: SDAE Open-World ───────────────────────────────────────────────────
print("\n" + "█"*60)
print("  SDAE  —  OPEN WORLD  (N+1 classes)")
print("█"*60)

# Update in_dim of first layer (same MAXLEN, just documenting clearly)
sdae_ow = build_sdae(SDAE_LAYERS, N_CLASSES_OPEN,
                     X_train_ow_flat, X_test_ow_flat,
                     pre_train=True)

sdae_ow.fit(
    X_train_ow_flat, y_train_ow_cat,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_split=0.1,
    callbacks=get_callbacks("sdae_open_best.h5"),
    verbose=1,
)
evaluate_open(sdae_ow, X_test_ow_flat, y_test_ow, y_test_ow_cat,
              N_CLASSES_OPEN, UNKNOWN_LABEL, "SDAE",
              prefix="tiktok_sdae")
